# Сборный проект №2 
# "Текущий уровень NPS телекоммуникационной компании"

## Описание проекта

**Заказчик исследования** — большая телекоммуникационная компания, которая оказывает услуги на территории всего СНГ. 

**Цель исследования:** определить текущий уровень потребительской лояльности, или `NPS` (от англ. Net Promoter Score), среди клиентов из России. 

**Задачи**:
- интерпретировать результаты NPS-опросов
- презентавать результаты NPS-опросов с помощью дашборда с его итогами.

***Примечания***: 
- Данные выгружены в SQLite — СУБД, в которой база данных представлена файлом. Путь к файлу: `/datasets/telecomm_csi.db`
- Преобразовывать данные с помощью Python нельзя — будут использованы только SQL-запросы. 

Работа будет выполнена в **4 этапа**:
1. Подключение к базе
2. Выгрузка данных и сохранение таблицы как CSV-файл
3. Анализ показателей и создание дашборда в Tableau
4. Визуализация показателей в дашборде и ответы на вопросы в презентации.

## Подключение к базе

Прежде чем начать работу, получим доступ к базе данных. Данные выгрузили в `SQLite` — СУБД, в которой база данных представлена файлом. Для подключения к такой базе достаточно иметь доступ к файлу с расширением `.db`.

**Чтобы подключиться к базе данных и сохранить данные в датафрейм в `pandas`, используем этот код**: 

In [1]:
# выгрузим необходимые библиотеки
import os
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

In [2]:
# путь к БД на локальном компьютере 
path_to_db_local = 'C:/Users/Мария/Desktop/Практикум/Датасеты/telecomm_csi.db'
# путь к БД на платформе
path_to_db_platform = '/datasets/telecomm_csi.db'

# итоговый путь к БД
path_to_db = None

# если путь на локальном компьютере ведёт к БД, то он становится итоговым
if os.path.exists(path_to_db_local):
    path_to_db = path_to_db_local
# иначе: если путь на платформе ведёт к БД, то он становится итоговым
elif os.path.exists(path_to_db_platform):
    path_to_db = path_to_db_platform
# иначе выводится сообщение о том, что файл не найден
else:
    raise Exception('Файл с базой данных SQLite не найден!')

# если итоговый путь не пустой
if path_to_db:
    # то создаём подключение к базе
    engine = create_engine(f'sqlite:///{path_to_db}', echo=False)

## Выгрузка данных

Теперь займемся подготовкой данных для построения дашборда. 

*Примечание: Преобразовывать данные с помощью Python нельзя — нужно использовать только SQL-запросы*. 

Соберем в одну витрину данные из разных таблиц. Эту витрину будем использовать для построения дашборда.   

Выгружать будем следующие поля:

1. `user_id` - идентификатор клиента,
2. `lt_day` - количество дней «жизни» клиента,
3. `is_new` - поле хранит информацию о том, является ли клиент новым,
4. `age` - возраст,
5. `gender_segment` - пол (для удобства работы с полем преобразуем значения в текстовый вид),
6. `os_name` - тип операционной системы,
7. `cpe_type_name` - тип устройства,
8. `country` - страна проживания,
9. `city` - город проживания,
10. `age_segment` - возрастной сегмент,
11. `traffic_segment` - сегмент по объёму потребляемого трафика,
12. `lifetime_segment` - сегмент по количеству дней «жизни»,
13. `nps_score` - оценка клиента в NPS-опросе,
14. `nps_group` - поле хранит информацию о том, к какой группе относится оценка клиента в опросе.

Напишем запрос, который выгрузит необходимые поля:

In [17]:
# Выгрузка необходимых полей из имеющихся таблиц
query = """
SELECT us.user_id,    
       us.lt_day, 
       CASE                                                              --определим, явялется ли клиент новым
           WHEN us.lt_day <= 365  THEN 'новый клиент'                    
           ELSE 'старый клиент'
       END AS is_new,
       CAST(us.age AS int4) as age,                                      --преобразуем возраст в целочисленный тип
       CASE 
           WHEN us.gender_segment = 1 THEN 'женщина'                     --для удобства работы с полем преобразуем значения 0/1 в текстовый вид
           ELSE 'мужчина'
       END AS gender_segment,
       us.os_name,
       us.cpe_type_name,
       loc.country,
       loc.city,
       SUBSTRING(ags.title, 4) AS age_segment,                           --удалим первые три символа из столбца (в трех таблицах идентичные действия)
       SUBSTRING(trs.title, 4) AS traffic_segment,
       SUBSTRING(lfs.title, 4) AS lifetime_segment,
       us.nps_score,
       CASE                                                              --разделим клиентов по результам опроса на уровень лояльности на 3 категории
           WHEN us.nps_score >= 0 AND us.nps_score <= 6  THEN 'критики'
           WHEN us.nps_score >= 7 AND us.nps_score <= 8  THEN 'нейтралы'
           ELSE 'cторонники'
       END AS nps_group
FROM user as us
LEFT JOIN location AS loc ON us.location_id = loc.location_id            --объеденим таблицы по первичному ключу
LEFT JOIN age_segment AS ags ON ags.age_gr_id = us.age_gr_id
LEFT JOIN traffic_segment AS trs ON trs.tr_gr_id = us.tr_gr_id
LEFT JOIN lifetime_segment AS lfs ON lfs.lt_gr_id = us.lt_gr_id;
""" 

In [18]:
# сохраняем полученный результат в переменную df и выводим на экран несколько первых строк
df = pd.read_sql(query, engine)
df.head()

,user_id,lt_day,is_new,age,gender_segment,os_name,cpe_type_name,country,city,age_segment,traffic_segment,lifetime_segment,nps_score,nps_group
0,A001A2,2320,старый клиент,45.0,женщина,ANDROID,SMARTPHONE,Россия,Уфа,45-54,1-5,36+,10,cторонники
1,A001WF,2344,старый клиент,53.0,мужчина,ANDROID,SMARTPHONE,Россия,Киров,45-54,1-5,36+,10,cторонники
2,A003Q7,467,старый клиент,57.0,мужчина,ANDROID,SMARTPHONE,Россия,Москва,55-64,20-25,13-24,10,cторонники
3,A004TB,4190,старый клиент,44.0,женщина,IOS,SMARTPHONE,Россия,РостовнаДону,35-44,0.1-1,36+,10,cторонники
4,A004XT,1163,старый клиент,24.0,мужчина,ANDROID,SMARTPHONE,Россия,Рязань,16-24,5-10,36+,10,cторонники


## Сохранение результатов выгрузки в таблице 

In [5]:
# Получившуюся таблицу сохраняем как CSV-файл.
df.to_csv('telecomm_csi_tableau.csv', index=False)

## Создание дашборда в Tableau и ответы на вопросы

[Ссылка на дашборд и презентацию](https://public.tableau.com/views/Book1_17201538110510/sheet21?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)